In [1]:
pip install azure-ai-ml azure-identity mltable mlflow azureml-mlflow

INFO: pip is looking at multiple versions of azureml-mlflow to determine which version is compatible with other requirements. This could take a while.
  Using cached azureml_mlflow-1.62.0.post1-py3-none-any.whl.metadata (2.9 kB)
  Using cached azureml_mlflow-1.62.0-py3-none-any.whl.metadata (2.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 27.9 MB/s  0:00:00
  Attempting uninstall: opentelemetry-api
    Found existing installation: opentelemetry-api 1.39.1
    Uninstalling opentelemetry-api-1.39.1:
      Successfully uninstalled opentelemetry-api-1.39.1
  Attempting uninstall: mlflow-skinny
    Found existing installation: mlflow-skinny 3.5.0
    Uninstalling mlflow-skinny-3.5.0:
      Successfully uninstalled mlflow-skinny-3.5.0━━━━━━━━━━━━━━━━ 1/3 [mlflow-skinny]
  Attempting uninstall: azureml-mlflow0m━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/3 [mlflow-skinny]
    Found existing installation: azureml-mlflow 1

In [2]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
from azure.identity import DefaultAzureCredential, InteractiveBrowserCredential
from azure.ai.ml import MLClient

# ── AUTHENTICATE ─────────────────────────────────────────────────────────────
# Tries environment/CLI/managed identity first; falls back to browser login
try:
    credential = DefaultAzureCredential()
    credential.get_token("https://management.azure.com/.default")
    print("✓ DefaultAzureCredential succeeded")
except Exception:
    credential = InteractiveBrowserCredential()
    print("✓ Falling back to InteractiveBrowserCredential")

✓ DefaultAzureCredential succeeded


In [3]:
ml_client = MLClient.from_config(credential=credential)

Found the config file in: /config.json
Class DeploymentTemplateOperations: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.


In [4]:
import os

# create a folder for the script files
script_folder = 'src'
os.makedirs(script_folder, exist_ok=True)
print(script_folder, 'folder created')

src folder created


In [5]:
%%writefile $script_folder/train-model-mlflow.py
# import libraries
import mlflow
import argparse
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.metrics import roc_curve
import matplotlib.pyplot as plt

def main(args):
    # read data
    df = get_data(args.training_data)

    # split data
    X_train, X_test, y_train, y_test = split_data(df)

    # train model
    model = train_model(args.reg_rate, X_train, X_test, y_train, y_test)

    # evaluate model
    eval_model(model, X_test, y_test)

# function that reads the data
def get_data(path):
    print("Reading data...")
    df = pd.read_csv(path)
    
    return df

# function that splits the data
def split_data(df):
    print("Splitting data...")
    X, y = df[['Pregnancies','PlasmaGlucose','DiastolicBloodPressure','TricepsThickness',
    'SerumInsulin','BMI','DiabetesPedigree','Age']].values, df['Diabetic'].values

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=0)

    return X_train, X_test, y_train, y_test

# function that trains the model
def train_model(reg_rate, X_train, X_test, y_train, y_test):
    mlflow.log_param("Regularization rate", reg_rate)
    print("Training model...")
    model = LogisticRegression(C=1/reg_rate, solver="liblinear").fit(X_train, y_train)

    return model

# function that evaluates the model
def eval_model(model, X_test, y_test):
    # calculate accuracy
    y_hat = model.predict(X_test)
    acc = np.average(y_hat == y_test)
    print('Accuracy:', acc)
    mlflow.log_metric("Accuracy", acc)

    # calculate AUC
    y_scores = model.predict_proba(X_test)
    auc = roc_auc_score(y_test,y_scores[:,1])
    print('AUC: ' + str(auc))
    mlflow.log_metric("AUC", auc)

    # plot ROC curve
    fpr, tpr, thresholds = roc_curve(y_test, y_scores[:,1])
    fig = plt.figure(figsize=(6, 4))
    # Plot the diagonal 50% line
    plt.plot([0, 1], [0, 1], 'k--')
    # Plot the FPR and TPR achieved by our model
    plt.plot(fpr, tpr)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.title('ROC Curve')
    plt.savefig("ROC-Curve.png")
    mlflow.log_artifact("ROC-Curve.png")    

def parse_args():
    # setup arg parser
    parser = argparse.ArgumentParser()

    # add arguments
    parser.add_argument("--training_data", dest='training_data',
                        type=str)
    parser.add_argument("--reg_rate", dest='reg_rate',
                        type=float, default=0.01)

    # parse args
    args = parser.parse_args()

    # return args
    return args

# run script
if __name__ == "__main__":
    # add space in logs
    print("\n\n")
    print("*" * 60)

    # parse args
    args = parse_args()

    # run main function
    main(args)

    # add space in logs
    print("*" * 60)
    print("\n\n")


Writing src/train-model-mlflow.py


In [6]:
from azure.ai.ml import command

# configure job

job = command(
    code="./src",
    command="python train-model-mlflow.py --training_data diabetes.csv",
    environment="AzureML-sklearn-1.5@latest",
    compute="aml-cluster",
    display_name="diabetes-train-mlflow",
    experiment_name="diabetes-training", 
    tags={"model_type": "LogisticRegression"}
    )

# submit job
returned_job = ml_client.create_or_update(job)
aml_url = returned_job.studio_url
print("Monitor your job at", aml_url)

Class AutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class AutoDeleteConditionSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseAutoDeleteSettingSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class IntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class ProtectionLevelSchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Class BaseIntellectualPropertySchema: This is an experimental class, and may change at any time. Please see https://aka.ms/azuremlexperimental for more information.
Uploading src (0.57 MBs): 100%|███████

Monitor your job at https://ml.azure.com/runs/sweet_plane_p0p636rh23?wsid=/subscriptions/735ef05d-6290-4c6c-bbb5-2e8ba6ecb52e/resourcegroups/rg-dp100-laec35969a22540f087/workspaces/mlw-dp100-laec35969a22540f087&tid=e43b0362-e932-4a58-9c0d-3071db3f47a8


Use MLflow to view and search for experiments

To list the jobs in the workspace, use the following command to list the experiments in the workspace:

In [7]:
import mlflow
experiments = mlflow.search_experiments()
for exp in experiments:
    print(exp.name)

diabetes-training


To retrieve a specific experiment, you can get it by its name:

In [8]:
experiment_name = "diabetes-training"
exp = mlflow.get_experiment_by_name(experiment_name)
print(exp)

<Experiment: artifact_location='', creation_time=1773937334929, experiment_id='6126117f-8aa3-47c1-bdae-56e2a1ab3ca7', last_update_time=None, lifecycle_stage='active', name='diabetes-training', tags={}>


Using an experiment name, you can retrieve all jobs of that experiment:

In [9]:
mlflow.search_runs(exp.experiment_id)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.AUC,metrics.Accuracy,params.Regularization rate,tags.mlflow.runName,tags.mlflow.rootRunId,tags.mlflow.user,tags.model_type
0,gentle_bird_rdhxthtc4t,6126117f-8aa3-47c1-bdae-56e2a1ab3ca7,FINISHED,,2026-03-19 16:24:43.781000+00:00,2026-03-19 16:26:11.623000+00:00,NaN,NaN,None,diabetes-train-script,gentle_bird_rdhxthtc4t,Alexander Castillo,None
1,sweet_plane_p0p636rh23,6126117f-8aa3-47c1-bdae-56e2a1ab3ca7,FINISHED,,2026-03-19 19:12:07.252000+00:00,2026-03-19 19:14:03.110000+00:00,0.848321,0.774,0.01,diabetes-train-mlflow,sweet_plane_p0p636rh23,Alexander Castillo,LogisticRegression


To more easily compare job runs and outputs, you can configure the search to order the results. For example, the following cell orders the results by start_time, and only shows a maximum of 2 results:

In [10]:
mlflow.search_runs(exp.experiment_id, order_by=["start_time DESC"], max_results=2)

,run_id,experiment_id,status,artifact_uri,start_time,end_time,metrics.AUC,metrics.Accuracy,params.Regularization rate,tags.mlflow.runName,tags.mlflow.user,tags.model_type,tags.mlflow.rootRunId
0,sweet_plane_p0p636rh23,6126117f-8aa3-47c1-bdae-56e2a1ab3ca7,FINISHED,,2026-03-19 19:12:07.252000+00:00,2026-03-19 19:14:03.110000+00:00,0.848321,0.774,0.01,diabetes-train-mlflow,Alexander Castillo,LogisticRegression,sweet_plane_p0p636rh23
1,gentle_bird_rdhxthtc4t,6126117f-8aa3-47c1-bdae-56e2a1ab3ca7,FINISHED,,2026-03-19 16:24:43.781000+00:00,2026-03-19 16:26:11.623000+00:00,NaN,NaN,None,diabetes-train-script,Alexander Castillo,None,gentle_bird_rdhxthtc4t


You can even create a query to filter the runs. Filter query strings are written with a simplified version of the SQL WHERE clause.

To filter, you can use two classes of comparators:

- Numeric comparators (metrics): =, !=, >, >=, <, and <=.
- String comparators (params, tags, and attributes): = and !=.

In [ ]:
query = "metrics.AUC > 0.8 and tags.model_type = 'LogisticRegression'"
mlflow.search_runs(exp.experiment_id, filter_string=query)